<div style="background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%); padding: 30px; border-radius: 10px; color: white; margin-bottom: 20px;">

<h1 style="color: white; margin: 0;">Stratospheric Aerosol Radiative Forcing</h1>
<h3 style="color: #c8d8f0; margin-top: 8px;">An interactive tutorial for climate researchers</h3>

<p style="margin-top: 15px; font-size: 15px;">
How does a thin layer of sulfate haze in the lower stratosphere cool the Earth? This notebook walks through the physics step by step, ending with a live playground where you can dial in your own conditions and see the answer update.
</p>

</div>

<div style="background: #f3e5f5; border-left: 6px solid #8e24aa; padding: 12px 18px; border-radius: 4px; margin: 10px 0;">

<b style="color: #4a148c;">Acknowledgments.</b>
The calculation presented here follows the simple-model formulation of <b>Jeffrey R. Pierce</b> (Colorado State University) and colleagues, as published in <i>Geophysical Research Letters</i>, <b>37</b>, L18805 (2010). We thank Jeff for making his original Python implementation publicly available and for pioneering the use of this simple-model chain in the stratospheric-aerosol-injection (SAI) literature. The underlying physics borrows from Bohren &amp; Huffman (1983) for Mie scattering, Chylek &amp; Wong (1995) for the simple two-stream radiative forcing equation, Wiscombe &amp; Grams (1976) for the upscatter fraction, and Tabazadeh <i>et al.</i> (1997) for the H<sub>2</sub>SO<sub>4</sub>/H<sub>2</sub>O equilibrium composition.

</div>

### Learning goals

By the end of this notebook you will:

1. Read the <b style="color:#2a5298;">Chylek &amp; Wong (1995)</b> radiative forcing equation and know what each term means physically.
2. See how <b style="color:#2a5298;">Mie scattering</b> produces size-dependent scattering efficiency $Q_\text{sca}(r)$ and asymmetry $g(r)$.
3. Understand the <b style="color:#2a5298;">upscatter fraction</b> $\beta(g)$ — the share of scattered light that actually returns to space.
4. Compute the <b style="color:#2a5298;">aerosol composition</b> in equilibrium with stratospheric water vapour.
5. Reproduce <b style="color:#2a5298;">Pierce (2010) Figure 1</b> — scattering cooling efficiency vs particle size.
6. Play with sliders for $T$, RH, refractive index, and atmospheric parameters and watch the answer respond.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import ipywidgets as widgets
from IPython.display import display, HTML

mpl.rcParams.update({
    'font.size': 11,
    'figure.dpi': 100,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.axisbelow': True,
    'figure.facecolor': 'white',
})

PALETTE = {
    'primary':   '#2a5298',
    'secondary': '#e07a5f',
    'accent':    '#81b29a',
    'warm':      '#f2cc8f',
    'deep':      '#3d405b',
    'rose':      '#c44569',
    'sky':       '#70a9a1',
    'gold':      '#f9c74f',
}

from tomas_jax.physics.radiative_forcing import (
    h2so4_equilibrium_wt, h2so4_solution_density,
    scattering_efficiency_vs_radius, upscatter_fraction,
    SOLAR_CONSTANT_PIERCE, TATM_STRATOSPHERIC,
    CLOUD_FRACTION_DEFAULT, ALBEDO_SURFACE_CLEARSKY,
    REFINDEX_SULFATE,
)
from tomas_jax.physics.bhmie import bhmie_qsca_jax

print('Pierce (2010) stratospheric defaults:')
print(f'  Solar constant S0  = {SOLAR_CONSTANT_PIERCE} W/m^2')
print(f'  Atm. transmittance = {TATM_STRATOSPHERIC}   (stratosphere: nothing above)')
print(f'  Cloud fraction A   = {CLOUD_FRACTION_DEFAULT}')
print(f'  Surface albedo R   = {ALBEDO_SURFACE_CLEARSKY}   (clear-sky global mean)')
print(f'  Refractive index   = {REFINDEX_SULFATE}    (~65 wt% H2SO4/H2O)')

## 1. Why do we care about radiative forcing?

<div style="background: #e8f5e9; border-left: 6px solid #2e7d32; padding: 12px 18px; border-radius: 4px; margin: 10px 0;">

<b style="color: #1b5e20;">The punchline.</b>
A thin haze of micron-scale sulfate particles in the lower stratosphere reflects a portion of incoming sunlight back to space before it reaches the ground — producing a <b>negative</b> (cooling) top-of-atmosphere energy anomaly. The magnitude of that cooling per unit of injected sulfur depends sensitively on <i>particle size</i>.

</div>

Large volcanic eruptions — e.g. Pinatubo in 1991 — inject several megatons of SO$_2$ into the stratosphere, where it oxidises to H$_2$SO$_4$ and forms micron-scale sulfate particles that cool the Earth by a few tenths of a degree for one to two years. **Stratospheric aerosol injection (SAI)** proposes doing this deliberately, at a sustained rate, to counteract some of the warming from anthropogenic CO$_2$.

Whether SAI makes sense depends on *how efficiently* an injected mass of sulfur cools the planet — which in turn depends on the size distribution produced by microphysics (nucleation, condensation, coagulation). Pierce *et al.* (2010) combined a microphysical model with the simple radiative forcing equation presented below to explore how injection rate and injection strategy shape that size distribution and hence the cooling achieved. This notebook focuses on the final link in that chain:

$$
\text{size distribution + composition} \;\longrightarrow\; \text{radiative forcing}.
$$

## 2. The Chylek &amp; Wong (1995) radiative forcing equation

For a thin, purely-scattering aerosol layer above a partly-cloudy Earth, the top-of-atmosphere radiative forcing is

$$
\large\boxed{\;\text{RF} \;=\; -\frac{S_0}{4}\, T_a^{\,2}\,(1-A)\,(1-R)^2\, 2\beta\, \tau\;}
$$

<table style="width:95%; border-collapse: collapse; margin: 15px 0;">
<thead>
<tr style="background:#2a5298; color:white;">
<th style="padding:8px 12px; text-align:left;">Symbol</th>
<th style="padding:8px 12px; text-align:left;">Meaning</th>
<th style="padding:8px 12px;">Pierce (2010)</th>
</tr>
</thead>
<tbody>
<tr style="background:#f5f9ff;"><td style="padding:8px;"><b>$S_0$</b></td><td style="padding:8px;">Solar constant [W/m²]. $S_0/4$ is the global-average insolation.</td><td style="padding:8px; text-align:center;"><b>1370</b></td></tr>
<tr><td style="padding:8px;"><b>$T_a$</b></td><td style="padding:8px;">Atmospheric transmittance <i>above</i> the aerosol layer. Stratospheric aerosol has nothing above it → $T_a = 1$. Tropospheric aerosol: $T_a \approx 0.85$.</td><td style="padding:8px; text-align:center;"><b>1.0</b></td></tr>
<tr style="background:#f5f9ff;"><td style="padding:8px;"><b>$A$</b></td><td style="padding:8px;">Cloud fraction. Clouds block the aerosol's reflection path; only the clear-sky fraction $(1-A)$ contributes.</td><td style="padding:8px; text-align:center;"><b>0.6</b></td></tr>
<tr><td style="padding:8px;"><b>$R$</b></td><td style="padding:8px;">Clear-sky surface albedo. Over bright surfaces upscattered light would have bounced back anyway, reducing net cooling. The $(1-R)^2$ factor counts two passes through the aerosol layer.</td><td style="padding:8px; text-align:center;"><b>0.15</b></td></tr>
<tr style="background:#f5f9ff;"><td style="padding:8px;"><b>$\beta$</b></td><td style="padding:8px;">Upscatter fraction. The share of scattered light that is actually redirected back toward space (vs forward/sideways).</td><td style="padding:8px; text-align:center;"><i>computed</i></td></tr>
<tr><td style="padding:8px;"><b>$\tau$</b></td><td style="padding:8px;">Scattering optical depth of the aerosol column.</td><td style="padding:8px; text-align:center;"><i>computed</i></td></tr>
</tbody>
</table>

<div style="background: #fff3e0; border-left: 6px solid #e65100; padding: 12px 18px; border-radius: 4px; margin: 10px 0;">

<b style="color: #bf360c;">Assumptions.</b>
(i) purely scattering aerosol — absorption neglected (valid for sulfate at visible wavelengths; not for black carbon); (ii) thin-layer limit — RF is linear in optical depth; (iii) globally-averaged solar geometry, albedo, and upscatter; (iv) no interaction with clouds or longwave radiation.

</div>

The two quantities we still need are **$\beta$** (from the particle's scattering phase function) and **$\tau$** (from the column-integrated scattering cross section). Both depend on **particle size** and **composition**. That is where Mie theory and the Tabazadeh parameterization come in.

## 3. Mie scattering — size matters

For a sphere of radius $r$ and complex refractive index $n$ illuminated by light of wavelength $\lambda$, Mie theory solves Maxwell's equations exactly and returns three dimensionless quantities in terms of the **size parameter** $x = 2\pi r / \lambda$:

| Quantity | Symbol | What it tells you |
|---|---|---|
| Scattering efficiency | $Q_\text{sca}$ | How effectively the particle deflects light, relative to its geometric cross section. |
| Extinction efficiency | $Q_\text{ext}$ | Total removal (scattering + absorption). For sulfate, $Q_\text{ext} \approx Q_\text{sca}$. |
| Asymmetry parameter | $g$ | Mean cosine of the scattering angle. $g=0$ isotropic; $g\to1$ pure forward. |

Three regimes show up clearly in the plot below:

<div style="background: #e3f2fd; border-left: 6px solid #1565c0; padding: 12px 18px; border-radius: 4px; margin: 10px 0;">

<b style="color: #0d47a1;">Three regimes of light scattering:</b><br>
<ul style="margin: 6px 0 0 0;">
<li><span style="background:#bbdefb; padding: 1px 8px; border-radius:3px;"><b>Rayleigh</b></span> &nbsp; $r \ll \lambda$: &nbsp; $Q_\text{sca} \propto r^4$ (tiny), nearly isotropic ($g \approx 0$). Why the sky is blue.</li>
<li><span style="background:#90caf9; padding: 1px 8px; border-radius:3px;"><b>Mie resonance</b></span> &nbsp; $r \sim \lambda$: &nbsp; oscillations, peak near $r \approx \lambda/2$. This is where stratospheric aerosol lives.</li>
<li><span style="background:#42a5f5; padding: 1px 8px; border-radius:3px; color:white;"><b>Geometric optics</b></span> &nbsp; $r \gg \lambda$: &nbsp; $Q_\text{sca} \to 2$ (the extinction paradox), strongly forward ($g \to 1$).</li>
</ul>

</div>

In [ ]:
wavelength = 500e-9  # 500 nm, near peak of the solar spectrum
refindex = complex(1.4, 1e-8)  # typical stratospheric sulfate

radii_mie = np.logspace(-9, -5, 150)  # 1 nm to 10 um
x_arr = 2.0 * np.pi * radii_mie / wavelength
Qext = np.array([float(bhmie_qsca_jax(float(x), refindex)[0]) for x in x_arr])
Qsca = np.array([float(bhmie_qsca_jax(float(x), refindex)[1]) for x in x_arr])
gsca = np.array([float(bhmie_qsca_jax(float(x), refindex)[2]) for x in x_arr])
radii_um = radii_mie * 1e6

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

ax = axes[0]
ax.axvspan(1e-3, wavelength * 1e6 / 3.0, alpha=0.18, color='#bbdefb',
           label='Rayleigh')
ax.axvspan(wavelength * 1e6 / 3.0, wavelength * 1e6 * 3.0, alpha=0.18,
           color='#90caf9', label='Mie')
ax.axvspan(wavelength * 1e6 * 3.0, 1e2, alpha=0.18, color='#42a5f5',
           label='Geometric')
ax.loglog(radii_um, Qsca, color=PALETTE['primary'], lw=2.4,
          label=r'$Q_{\mathrm{sca}}$')
ax.loglog(radii_um, Qext, color=PALETTE['rose'], lw=2.4, ls='--',
          label=r'$Q_{\mathrm{ext}}$')
ax.axhline(2.0, color=PALETTE['deep'], ls=':', alpha=0.7,
           label=r'$Q\to 2$')
ax.set_xlim(1e-3, 10)
ax.set_ylim(1e-8, 10)
ax.set_xlabel(r'Particle radius [$\mu$m]')
ax.set_ylabel('Efficiency (dimensionless)')
ax.set_title('Scattering efficiency vs particle size')
ax.legend(loc='lower right', framealpha=0.9, fontsize=9)

ax = axes[1]
points = np.array([radii_um, gsca]).T.reshape(-1, 1, 2)
segments = np.concatenate([points[:-1], points[1:]], axis=1)
from matplotlib.collections import LineCollection
lc = LineCollection(segments, cmap='plasma', array=gsca, lw=3)
ax.add_collection(lc)
ax.set_xscale('log')
ax.set_xlim(1e-3, 10); ax.set_ylim(-0.05, 1.05)
ax.set_xlabel(r'Particle radius [$\mu$m]')
ax.set_ylabel('Asymmetry $g$  (0 = isotropic, 1 = forward)')
ax.set_title('Angular character of the scattering')
ax.axhline(0.0, color='gray', ls=':', alpha=0.5)
ax.axhline(1.0, color='gray', ls=':', alpha=0.5)
cbar = plt.colorbar(lc, ax=ax, shrink=0.85, pad=0.02)
cbar.set_label('$g$', rotation=0, labelpad=12)

fig.suptitle(f'Mie scattering for a sulfate sphere  '
             f'($\\lambda$={wavelength*1e9:.0f} nm, $n={refindex.real}+{refindex.imag:.0e}i$)',
             y=1.02, fontsize=12)
fig.tight_layout()

## 4. Upscatter fraction — what share comes back?

Scattering sends photons in every direction, but only photons that ultimately return to space **above the aerosol layer** contribute to cooling. The fraction that does so is the **upscatter fraction** $\beta$, defined for a given asymmetry parameter $g$ and solar zenith angle $\theta_0$ by

$$
\beta(g, \theta_0) \;=\; \frac{1}{2\pi} \int_0^{2\pi}\!\!\int_{\pi/2}^{\pi} p_\mathrm{HG}(g,\cos\Theta)\, \sin\theta\,\mathrm{d}\theta\,\mathrm{d}\phi ,
$$

integrating the Henyey–Greenstein phase function $p_\mathrm{HG}$ over the upward hemisphere (Wiscombe &amp; Grams 1976, eqn 22).

<div style="background: #e8f5e9; border-left: 6px solid #2e7d32; padding: 12px 18px; border-radius: 4px; margin: 10px 0;">

<b style="color:#1b5e20;">Physical intuition.</b>
Overhead sun, isotropic scatterer ($g=0$) ⇒ exactly half goes up: $\beta = 0.5$. A forward-peaked scatterer ($g \to 1$) sends almost everything downward: $\beta \to 0$. Stratospheric sulfate typically has $g \approx 0.6$–0.8 at visible wavelengths, so only about 20–30% of scattered photons escape back to space. The glancing geometry of low-sun latitudes improves the odds — more on that next.

</div>

In [ ]:
g_sweep = np.linspace(0.0, 0.99, 60)
sza_values_deg = [0, 30, 60, 75, 85]
sza_colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(sza_values_deg)))

fig, ax = plt.subplots(figsize=(9, 5))

ax.axhline(0.5, color='gray', ls=':', alpha=0.7)
ax.text(0.01, 0.51, 'isotropic ($\\beta=0.5$)', fontsize=9, color='gray')

for sza_deg, col in zip(sza_values_deg, sza_colors):
    sza_rad = np.deg2rad(sza_deg)
    beta_vals = np.array([float(upscatter_fraction(g, sza_rad)) for g in g_sweep])
    ax.plot(g_sweep, beta_vals, color=col, lw=2.4,
            label=f'sun at ${sza_deg}^\\circ$ from zenith')

ax.axvspan(0.6, 0.8, color=PALETTE['gold'], alpha=0.22)
ax.text(0.70, 0.93, 'typical sulfate', ha='center', color='#b8860b',
        fontsize=10, fontweight='bold')

ax.set_xlabel('Asymmetry parameter $g$')
ax.set_ylabel(r'Upscatter fraction $\beta$')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('Fraction of scattered light that escapes back to space')
ax.legend(loc='upper right', framealpha=0.95, fontsize=10)
fig.tight_layout()

print('Low-sun geometry (large solar zenith angle) gives HIGHER β for the same g,')
print('because photons are scattered closer to horizontal — easier to escape upward.')

## 5. Aerosol composition — it's not pure acid

A stratospheric sulfate particle is really a **liquid solution of sulfuric acid and water**, in equilibrium with the ambient water vapour. The weight-percent H$_2$SO$_4$ swings dramatically with temperature and humidity:

- **Cold + dry** (195 K, 1% RH): ~75 wt% H$_2$SO$_4$ → dense, concentrated acid.
- **Warmer + moister** (230 K, 10% RH): ~55 wt% → much more water, lower density, lower refractive index, but *more particle mass per unit sulfur*.

Tabazadeh *et al.* (1997) parameterized the equilibrium vapour pressure of water over the solution as

$$
\ln P_\mathrm{H_2O}(T,w) \;=\; a(w) + \frac{b(w)}{T} + \frac{c(w)}{T^2}
$$

for 15 weight percents $w \in \{10, 15, \ldots, 80\}\%$. Given $T$ and RH, we invert the relation to find the $w$ that matches the ambient $P_\mathrm{H_2O}$.

<div style="background: #fce4ec; border-left: 6px solid #ad1457; padding: 12px 18px; border-radius: 4px; margin: 10px 0;">

<b style="color:#880e4f;">Why it matters for RF.</b>
Pierce (2010) assumed 75 wt% in their figures (which corresponds to RH&nbsp;&lt;&nbsp;1% — very dry). Typical lower-stratospheric RH of 2–10% gives 55–70 wt%. That difference of 10–20 percentage points in water content changes the solution density by ~10%, the refractive index slightly, and — most importantly — the cooling efficiency <i>per megaton of sulfur</i>, because wetter aerosol delivers more scattering material per Mt-S.

</div>

In [ ]:
T_grid = np.linspace(195, 255, 50)
RH_grid = np.linspace(1, 20, 50)
wt_heat = np.zeros((len(RH_grid), len(T_grid)))
rho_heat = np.zeros_like(wt_heat)
for i, rh in enumerate(RH_grid):
    for j, T in enumerate(T_grid):
        w = h2so4_equilibrium_wt(T, rh)
        wt_heat[i, j] = w
        rho_heat[i, j] = h2so4_solution_density(w)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
im = ax.pcolormesh(T_grid, RH_grid, wt_heat, cmap='viridis', shading='auto',
                   vmin=40, vmax=80)
cs = ax.contour(T_grid, RH_grid, wt_heat,
                levels=[50, 55, 60, 65, 70, 75], colors='white',
                linewidths=1.0, alpha=0.8)
ax.clabel(cs, inline=True, fontsize=8.5, fmt='%d%%')
ax.set_xlabel('Temperature $T$ [K]')
ax.set_ylabel('Relative humidity [%]')
ax.set_title('H$_2$SO$_4$ weight percent')
cb = plt.colorbar(im, ax=ax, label='wt% H$_2$SO$_4$')
ax.plot(220, 5, marker='*', ms=22, color='white',
        markeredgecolor=PALETTE['rose'], mew=2.0, label='T=220 K, RH=5%')
ax.legend(loc='upper right', framealpha=0.95)

ax = axes[1]
im = ax.pcolormesh(T_grid, RH_grid, rho_heat, cmap='plasma', shading='auto')
cs = ax.contour(T_grid, RH_grid, rho_heat,
                levels=[1400, 1450, 1500, 1550, 1600, 1650], colors='white',
                linewidths=1.0, alpha=0.8)
ax.clabel(cs, inline=True, fontsize=8.5, fmt='%d')
ax.set_xlabel('Temperature $T$ [K]')
ax.set_ylabel('Relative humidity [%]')
ax.set_title('Solution density')
plt.colorbar(im, ax=ax, label=r'$\rho$ [kg/m$^3$]')
ax.plot(220, 5, marker='*', ms=22, color='white',
        markeredgecolor=PALETTE['rose'], mew=2.0)

fig.suptitle('Binary H$_2$SO$_4$/H$_2$O equilibrium (Tabazadeh et al. 1997)',
             fontsize=13, y=1.02)
fig.tight_layout()

w0 = h2so4_equilibrium_wt(220.0, 5.0)
r0 = h2so4_solution_density(w0)
print(f'Marker point  (T=220 K, RH=5%):  {w0:.1f} wt% H2SO4, density {r0:.0f} kg/m^3')

## 6. Single wavelength vs the full solar spectrum

The solar spectrum spans roughly 300–2500 nm. Pierce (2010) used a single representative wavelength (500 nm), which captures the peak of the solar spectrum but **overestimates** the peak radiative forcing because Mie resonance oscillations are sharp in $r$ and smooth out when integrated over the full spectrum.

Below we compare:

- <span style="color:#c44569; font-weight:bold;">Single wavelength</span> at 500 nm (one Mie calculation per radius).
- <span style="color:#2a5298; font-weight:bold;">Full spectrum</span> weighted by a 5778 K Planck function over 300–2500 nm, 50 bands.

<div style="background: #ffebee; border-left: 6px solid #c62828; padding: 12px 18px; border-radius: 4px; margin: 10px 0;">

<b style="color:#b71c1c;">Practical difference.</b>
Spectral weighting reduces peak cooling efficiency by roughly <b>30–40%</b> relative to the single-wavelength calculation, and shifts the optimum size from $\approx 0.19\,\mu$m to $\approx 0.23\,\mu$m. Any quantitative statement about "optimal injection size" or "climate sensitivity to sulfate" that uses the single-wavelength result is biased high.

</div>

In [ ]:
T_ref = 220.0
RH_ref = 5.0
wt_ref = h2so4_equilibrium_wt(T_ref, RH_ref)
rho_ref = h2so4_solution_density(wt_ref)

shared_kw = dict(
    refindex=REFINDEX_SULFATE, density=rho_ref,
    solar_constant=SOLAR_CONSTANT_PIERCE,
    Tatm=TATM_STRATOSPHERIC,
    albedo=ALBEDO_SURFACE_CLEARSKY,
    cloud_fraction=CLOUD_FRACTION_DEFAULT,
    n_radii=80, r_min=5e-9, r_max=10e-6,
)

radii_s, rf_single = scattering_efficiency_vs_radius(
    wavelength=500e-9, spectral=False, **shared_kw)
radii_b, rf_spec = scattering_efficiency_vs_radius(
    spectral=True, n_wavelengths=50, **shared_kw)

s_frac = wt_ref / 100.0 * 32.0 / 98.0
KG_PER_MTS = 1e9 / s_frac
cool_single = -rf_single * KG_PER_MTS
cool_spec = -rf_spec * KG_PER_MTS

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.fill_between(radii_s * 1e6, 0, cool_single, color=PALETTE['rose'],
                alpha=0.15)
ax.fill_between(radii_b * 1e6, 0, cool_spec, color=PALETTE['primary'],
                alpha=0.15)
ax.semilogx(radii_s * 1e6, cool_single, color=PALETTE['rose'], lw=2.8,
            label=f'Single $\\lambda$ = 500 nm   (peak {cool_single.max():.2f})')
ax.semilogx(radii_b * 1e6, cool_spec, color=PALETTE['primary'], lw=2.8,
            label=f'Spectral Planck-weighted   (peak {cool_spec.max():.2f})')

ipk_s = int(np.argmax(cool_single))
ipk_b = int(np.argmax(cool_spec))
ax.axvline(radii_s[ipk_s]*1e6, color=PALETTE['rose'], ls=':', alpha=0.5)
ax.axvline(radii_b[ipk_b]*1e6, color=PALETTE['primary'], ls=':', alpha=0.5)

ax.set_xlim(5e-3, 10)
ax.set_xlabel(r'Particle radius [$\mu$m]')
ax.set_ylabel('Cooling efficiency\n[W m$^{-2}$ per Mt-S]')
ax.set_title(f'Single $\\lambda$ vs spectral Mie  (T={T_ref:.0f} K, RH={RH_ref:.0f}%, '
             f'{wt_ref:.0f} wt%, $\\rho$={rho_ref:.0f} kg/m$^3$)')
ax.legend(loc='upper right', framealpha=0.95, fontsize=11)
fig.tight_layout()

print(f'Peak radius  single λ:  {radii_s[ipk_s]*1e6:.3f} μm    spectral: {radii_b[ipk_b]*1e6:.3f} μm')
print(f'Peak magnitude ratio (spectral / single): '
      f'{cool_spec.max()/cool_single.max():.2f}x  '
      f'({(1-cool_spec.max()/cool_single.max())*100:.0f}% reduction)')

## 7. Reproducing Pierce (2010) Figure 1

Here is the punchline figure. The cooling efficiency per **megaton of sulfur** is plotted vs particle size for several stratospheric humidities, each using its self-consistent Tabazadeh composition and density. Curves are produced with full spectral Mie.

<div style="background: #fff8e1; border-left: 6px solid #f57f17; padding: 12px 18px; border-radius: 4px; margin: 10px 0;">

<b style="color:#e65100;">Read this figure.</b>
Every curve has a clear optimum near <b>0.2–0.3 μm radius</b>. Particles much smaller than that are in the Rayleigh regime and barely scatter; particles much larger are strongly forward-scattering ($g \to 1$, so $\beta \to 0$) and also have less cross-section per unit mass. <b>The entire rationale for microphysics-informed SAI analysis is to understand where in this curve the injected aerosol actually ends up — and to inject in a way that keeps particles small.</b>

</div>

In [ ]:
T_strat = 220.0
rh_values = [2, 5, 10, 20]
rh_colors = plt.cm.cool(np.linspace(0.15, 0.9, len(rh_values)))

fig, ax = plt.subplots(figsize=(11, 6))

for rh, col in zip(rh_values, rh_colors):
    wt = h2so4_equilibrium_wt(T_strat, rh)
    rho = h2so4_solution_density(wt)
    r_vals, rf = scattering_efficiency_vs_radius(
        refindex=REFINDEX_SULFATE, density=rho,
        solar_constant=SOLAR_CONSTANT_PIERCE,
        Tatm=TATM_STRATOSPHERIC,
        albedo=ALBEDO_SURFACE_CLEARSKY,
        cloud_fraction=CLOUD_FRACTION_DEFAULT,
        spectral=True, n_wavelengths=50,
        n_radii=80, r_min=5e-9, r_max=10e-6,
    )
    s_frac = wt / 100.0 * 32.0 / 98.0
    cool = -rf * (1e9 / s_frac)
    ipk = int(np.argmax(cool))
    ax.semilogx(r_vals * 1e6, cool, color=col, lw=2.6,
                label=f'RH={rh:>2d}%  ({wt:.0f} wt%, $\\rho$={rho:.0f})   '
                      f'peak {cool.max():.2f} W m$^{{-2}}$ / Mt-S @ {r_vals[ipk]*1e6:.2f} $\\mu$m')
    ax.plot(r_vals[ipk]*1e6, cool.max(), marker='o', ms=11, color=col,
            markeredgecolor='white', mew=1.5, zorder=5)

ax.axvspan(0.15, 0.35, color=PALETTE['gold'], alpha=0.12)
ax.text(0.23, ax.get_ylim()[1]*0.03, 'sweet spot', ha='center',
        color='#b8860b', fontsize=10, fontweight='bold')

ax.set_xlim(5e-3, 10)
ax.set_xlabel(r'Particle radius [$\mu$m]')
ax.set_ylabel('Cooling efficiency\n[W m$^{-2}$ per Mt-S]')
ax.set_title(f'Scattering cooling efficiency vs particle size  (T = {T_strat:.0f} K)')
ax.legend(loc='upper right', fontsize=9.5, framealpha=0.95)
fig.tight_layout()

## 8. Your turn — interactive playground

<div style="background: #ede7f6; border-left: 6px solid #5e35b1; padding: 12px 18px; border-radius: 4px; margin: 10px 0;">

<b style="color:#311b92;">Dial it in.</b>
Move any slider below and everything — composition, density, refractive index, Pierce-SI parameters — recomputes and redraws. Useful experiments:
<ul style="margin:6px 0 0 0;">
<li>Cool the stratosphere: drop $T$ from 220 K to 195 K. Watch wt% climb and peak cooling drop.</li>
<li>Flip from stratospheric to tropospheric: set $T_a = 0.85$ and $A = 0.6$. Cooling halves.</li>
<li>Try a bright surface ($R = 0.6$, e.g. over ice). $(1-R)^2$ kills the forcing.</li>
<li>Set <b># λ = 1</b> to recover the single-wavelength curve. Peak jumps up ~40%.</li>
</ul>

</div>

In [ ]:
def plot_rf_interactive(T, RH, n_real, n_imag_log, solar_constant,
                        Tatm, cloud_fraction, albedo, n_wavelengths):
    wt = h2so4_equilibrium_wt(T, RH)
    rho = h2so4_solution_density(wt)
    nref = complex(n_real, 10.0 ** n_imag_log)
    spectral = n_wavelengths > 1

    radii, rf = scattering_efficiency_vs_radius(
        wavelength=500e-9, refindex=nref, density=rho,
        solar_constant=solar_constant, Tatm=Tatm,
        albedo=albedo, cloud_fraction=cloud_fraction,
        spectral=spectral, n_wavelengths=int(n_wavelengths),
        n_radii=80, r_min=5e-9, r_max=10e-6,
    )
    s_frac = wt / 100.0 * 32.0 / 98.0
    cool = -rf * (1e9 / s_frac)
    ipk = int(np.argmax(cool))
    peak_val = cool[ipk]
    peak_r_um = radii[ipk] * 1e6

    fig = plt.figure(figsize=(13.5, 5.0))
    gs = fig.add_gridspec(1, 3, width_ratios=[0.9, 0.9, 2.6], wspace=0.35)

    ax1 = fig.add_subplot(gs[0, 0])
    ax1.bar(['wt%'], [wt], color=PALETTE['primary'], width=0.5)
    ax1.set_ylim(0, 85)
    ax1.set_title(f'T={T:.0f} K, RH={RH:.0f}%')
    ax1.set_ylabel('H$_2$SO$_4$ wt%')
    ax1.text(0, wt + 2, f'{wt:.1f}%', ha='center', fontsize=12,
             fontweight='bold', color=PALETTE['primary'])
    ax1.set_xticks([]); ax1.grid(False, axis='x')

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.bar([r'$\rho$'], [rho], color=PALETTE['secondary'], width=0.5)
    ax2.set_ylim(1200, 1800)
    ax2.set_title('Solution density')
    ax2.set_ylabel(r'kg/m$^3$')
    ax2.text(0, rho + 20, f'{rho:.0f}', ha='center', fontsize=12,
             fontweight='bold', color=PALETTE['secondary'])
    ax2.set_xticks([]); ax2.grid(False, axis='x')

    ax3 = fig.add_subplot(gs[0, 2])
    ax3.fill_between(radii * 1e6, 0, cool, color=PALETTE['primary'], alpha=0.18)
    ax3.semilogx(radii * 1e6, cool, color=PALETTE['primary'], lw=2.6)
    ax3.plot(peak_r_um, peak_val, marker='o', ms=12,
             color=PALETTE['secondary'], markeredgecolor='white', mew=1.8, zorder=5)
    ax3.axvline(peak_r_um, color=PALETTE['secondary'], ls=':', alpha=0.6)
    ax3.annotate(f'peak {peak_val:.2f} W m$^{{-2}}$ / Mt-S\n@ r = {peak_r_um:.2f} $\\mu$m',
                 xy=(peak_r_um, peak_val),
                 xytext=(0.98, 0.92), textcoords='axes fraction',
                 ha='right', va='top', fontsize=10.5,
                 bbox=dict(boxstyle='round,pad=0.45', fc='white',
                           ec=PALETTE['secondary'], lw=1.5),
                 arrowprops=dict(arrowstyle='->', color=PALETTE['secondary']))
    ax3.set_xlim(5e-3, 10)
    ax3.set_xlabel(r'Particle radius [$\mu$m]')
    ax3.set_ylabel('Cooling efficiency\n[W m$^{-2}$ per Mt-S]')
    mode = 'spectral' if spectral else 'single $\\lambda$ = 500 nm'
    ax3.set_title(f'RF vs size   ({mode},  $n$ = {n_real:.2f}{(10.0**n_imag_log):+.0e}i)')

    plt.show()


style = {'description_width': '130px'}
layout = widgets.Layout(width='460px')

interact = widgets.interact(
    plot_rf_interactive,
    T=widgets.FloatSlider(value=220, min=185, max=260, step=2,
                          description='Temperature [K]',
                          continuous_update=False, style=style, layout=layout),
    RH=widgets.FloatSlider(value=5, min=1, max=20, step=1,
                           description='Relative humidity [%]',
                           continuous_update=False, style=style, layout=layout),
    n_real=widgets.FloatSlider(value=1.4, min=1.3, max=1.6, step=0.01,
                               description='Refractive n (real)',
                               continuous_update=False, style=style, layout=layout),
    n_imag_log=widgets.FloatSlider(value=-8, min=-9, max=-2, step=0.5,
                                   description='log₁₀ n (imag)',
                                   continuous_update=False, style=style, layout=layout),
    solar_constant=widgets.FloatSlider(value=1370, min=1360, max=1380, step=1,
                                       description='Solar S₀ [W/m²]',
                                       continuous_update=False, style=style, layout=layout),
    Tatm=widgets.FloatSlider(value=1.0, min=0.85, max=1.0, step=0.01,
                             description='Atm. Tₐ',
                             continuous_update=False, style=style, layout=layout),
    cloud_fraction=widgets.FloatSlider(value=0.6, min=0.0, max=0.8, step=0.05,
                                       description='Cloud fraction A',
                                       continuous_update=False, style=style, layout=layout),
    albedo=widgets.FloatSlider(value=0.15, min=0.05, max=0.7, step=0.01,
                               description='Surface albedo R',
                               continuous_update=False, style=style, layout=layout),
    n_wavelengths=widgets.Dropdown(
        options=[('1  (single λ = 500 nm)', 1), ('10', 10), ('30', 30), ('50', 50)],
        value=30, description='# wavelengths',
        style=style, layout=layout),
);

## References

<div style="background:#f5f5f5; border-radius: 6px; padding: 15px 20px; margin: 10px 0;">

<ol style="margin:0; padding-left: 20px;">
<li style="margin-bottom: 6px;"><b>Bohren, C. F.</b> and <b>D. R. Huffman</b> (1983), <i>Absorption and Scattering of Light by Small Particles</i>, Wiley-Interscience.</li>
<li style="margin-bottom: 6px;"><b>Chylek, P.</b> and <b>J. Wong</b> (1995), Effect of absorbing aerosols on global radiation budget, <i>Geophysical Research Letters</i>, <b>22</b>, 929–931.</li>
<li style="margin-bottom: 6px;"><b>Pierce, J. R.</b>, D. K. Weisenstein, P. Heckendorn, T. Peter, and D. W. Keith (2010), Efficient formation of stratospheric aerosol for climate engineering, <i>Geophysical Research Letters</i>, <b>37</b>, L18805, <a href="https://doi.org/10.1029/2010GL043975">doi:10.1029/2010GL043975</a>.</li>
<li style="margin-bottom: 6px;"><b>Tabazadeh, A.</b>, O. B. Toon, S. L. Clegg, and P. Hamill (1997), A new parameterization of H<sub>2</sub>SO<sub>4</sub>/H<sub>2</sub>O aerosol composition: Atmospheric implications, <i>Geophysical Research Letters</i>, <b>24</b>(15), 1931–1934.</li>
<li><b>Wiscombe, W. J.</b> and <b>G. W. Grams</b> (1976), The backscattered fraction in two-stream approximations, <i>Journal of the Atmospheric Sciences</i>, <b>33</b>, 2440–2451.</li>
</ol>

</div>

<div style="color:#666; font-size: 11px; margin-top: 20px;">Prepared for CloudHub climate-analysis researchers. Questions or suggestions: <a href="mailto:ali@reflective.org">ali@reflective.org</a>.</div>